In [ ]:
# =============================================================================
# CONFIGURATION & IMPORTS
# =============================================================================

# Configuration
TRIAL_NUMBER = 10  # Trial to analyze (has real movement data)
TIME_BIN_SIZE = 0.02  # seconds (20 ms bins)
SAMPLING_RATE = 30000  # Hz
SPIKE_CHANNELS = [0, 1, 2, 3, 6, 32, 39, 40, 41, 42, 46, 49, 53, 67, 68, 73, 74, 75, 76, 77, 84]

# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Modular imports
from utils.diagnostics import (
    diagnose_trial_data, 
    check_behavioral_data_availability,
    find_trials_with_movement,
    run_comprehensive_diagnostic
)
from utils.analysis import (
    FeatureAnalyzer,
    print_feature_summary,
    analyze_behavioral_correlation,
    get_trial_quality_score
)
from utils.visualization import (
    plot_neural_behavioral_sync,
    plot_feature_overview,
    plot_channel_comparison,
    plot_channel_detail,
    plot_behavior_raster_psth
)

# Set up plotting
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print(f"📊 Configuration: Trial {TRIAL_NUMBER}, {TIME_BIN_SIZE*1000:.0f}ms bins, {len(SPIKE_CHANNELS)} channels")


In [ ]:
# Additional imports for visualization functions
from utils.analysis import FeatureAnalyzer
from utils.visualization import plot_behavior_raster_psth, plot_multi_trial_raster_comparison


In [ ]:
# Run comprehensive diagnostic
diagnostic_results = run_comprehensive_diagnostic(TRIAL_NUMBER, SAMPLING_RATE)

# Extract results
if diagnostic_results['success']:
    trial_data = diagnostic_results['trial_data']
    validation = diagnostic_results['validation'] 
    movement_trials = diagnostic_results['movement_trials']
    
    print(f"\n🎯 Recommended trials with good movement:")
    print(f"   {movement_trials[:10]}")  # Show first 10 trials with movement
else:
    print("❌ Diagnostic failed!")
    trial_data = None


In [ ]:
# Initialize feature analyzer
analyzer = FeatureAnalyzer(SPIKE_CHANNELS)

# Load and extract features
trial_data, features = analyzer.load_and_extract_features(
    trial_number=TRIAL_NUMBER,
    time_bin_size=TIME_BIN_SIZE,
    sampling_rate=SAMPLING_RATE
)

# Print feature summary
if features is not None:
    print_feature_summary(features, SPIKE_CHANNELS)
    
    # Get most active channels
    top_channels = analyzer.find_most_active_channels(features, top_n=5)
    print(f"\n🔥 Most active channels for analysis: {top_channels}")
    
    # Get trial quality score
    quality = get_trial_quality_score(trial_data, features)
    print(f"\n⭐ Trial quality: {quality['assessment'].upper()} (score: {quality['overall_score']:.1f}/100)")
else:
    print("❌ Feature extraction failed!")


In [ ]:
# 1. Synchronized neural and behavioral visualization
if trial_data is not None and features is not None:
    plot_neural_behavioral_sync(
        trial_data=trial_data,
        features=features,
        spike_channels=SPIKE_CHANNELS,
        trial_number=TRIAL_NUMBER,
        figsize=(15, 12)
    )


In [ ]:
# 2. Feature overview plot
if features is not None:
    plot_feature_overview(
        features=features,
        spike_channels=SPIKE_CHANNELS,
        trial_data=trial_data,
        trial_number=TRIAL_NUMBER,
        n_channels=8,
        figsize=(15, 12)
    )


In [ ]:
# 3. Compare most active channels
if features is not None and 'top_channels' in locals():
    # Take top 4 channels for comparison
    channels_to_compare = top_channels[:4]
    
    plot_channel_comparison(
        features=features,
        spike_channels=SPIKE_CHANNELS,
        channel_list=channels_to_compare,
        trial_data=trial_data,
        figsize=(12, 10)
    )


In [ ]:
# 4. Detailed analysis of the most active channel
if features is not None and 'top_channels' in locals():
    most_active_channel = top_channels[0]
    
    plot_channel_detail(
        features=features,
        spike_channels=SPIKE_CHANNELS,
        channel_number=most_active_channel,
        trial_data=trial_data,
        figsize=(12, 12)
    )


In [ ]:
# Analyze behavioral correlation
if trial_data is not None and features is not None:
    correlation_results = analyze_behavioral_correlation(trial_data, features)
    
    print("🔗 NEURAL-BEHAVIORAL CORRELATION ANALYSIS:")
    print("=" * 50)
    
    if correlation_results['success']:
        print(f"✅ Correlation coefficient: {correlation_results['correlation']:.3f}")
        print(f"   Neural activity peak: {correlation_results['neural_activity_peak']:.3f}")
        print(f"   Velocity peak: {correlation_results['velocity_peak']:.3f}")
        print(f"   Samples analyzed: {correlation_results['neural_samples']}")
    else:
        print("⚠️  Correlation analysis not possible (no movement data)")
    
    # Get detailed channel statistics
    channel_stats = analyzer.get_channel_statistics(features, channel_indices=None)
    print(f"\n📊 Detailed statistics available for {len(channel_stats)} channels")
    
    # Compare top channels
    if 'top_channels' in locals():
        comparison = analyzer.compare_channels(features, top_channels[:3])
        print(f"\n🏆 Channel comparison results:")
        if 'spike_band' in comparison['features']:
            most_active = comparison['features']['spike_band']['most_active_channel']
            print(f"   Most active channel (spike band): {most_active}")
        if 'threshold' in comparison['features']:
            most_crossings = comparison['features']['threshold']['most_active_channel']
            print(f"   Most active channel (crossings): {most_crossings}")


In [ ]:
# =============================================================================
# INTERACTIVE EXPLORATION SECTION
# =============================================================================

# Change these parameters to explore different aspects:
EXPLORE_TRIAL = 10  # Try different trials from the movement_trials list
EXPLORE_CHANNELS = [0, 1, 2, 3]  # Try different channel combinations
EXPLORE_CHANNEL_DETAIL = 0  # Single channel for detailed analysis

print("🔍 INTERACTIVE EXPLORATION PARAMETERS:")
print(f"   Trial: {EXPLORE_TRIAL}")
print(f"   Channels for comparison: {EXPLORE_CHANNELS}")
print(f"   Channel for detailed analysis: {EXPLORE_CHANNEL_DETAIL}")
print(f"   Available trials with movement: {movement_trials[:10] if 'movement_trials' in locals() else 'Run diagnostics first'}")

# Quick analysis of a different trial
if EXPLORE_TRIAL != TRIAL_NUMBER:
    print(f"\n🔍 Quick analysis of trial {EXPLORE_TRIAL}:")
    
    # Load and extract features for exploration trial
    explore_trial_data, explore_features = analyzer.load_and_extract_features(
        trial_number=EXPLORE_TRIAL,
        time_bin_size=TIME_BIN_SIZE,
        sampling_rate=SAMPLING_RATE
    )
    
    if explore_features is not None:
        # Quick feature overview
        plot_feature_overview(
            features=explore_features,
            spike_channels=SPIKE_CHANNELS,
            trial_data=explore_trial_data,
            trial_number=EXPLORE_TRIAL,
            n_channels=4,
            figsize=(12, 10)
        )
        
        # Quality assessment
        explore_quality = get_trial_quality_score(explore_trial_data, explore_features)
        print(f"⭐ Trial {EXPLORE_TRIAL} quality: {explore_quality['assessment'].upper()} (score: {explore_quality['overall_score']:.1f}/100)")
else:
    print(f"\n💡 To explore a different trial, change EXPLORE_TRIAL to one of: {movement_trials[:5] if 'movement_trials' in locals() else 'Run diagnostics first'}")


In [ ]:
# 🔥 Extract Most Active Channels
# Define most_active_channels for use in visualizations

# Create analyzer and find most active channels
analyzer = FeatureAnalyzer(SPIKE_CHANNELS)
most_active_channels = analyzer.find_most_active_channels(features, top_n=10)

print(f"🔥 Most active channels: {most_active_channels}")
print(f"   Selected {len(most_active_channels)} channels for detailed analysis")


In [ ]:
# Import spike detection module
from utils.spike_detection import SpikeDetector, create_raster_comparison

# Initialize spike detector
spike_detector = SpikeDetector(sampling_rate=SAMPLING_RATE)

print("🔬 Spike Detection Comparison Module Loaded!")
print(f"   Sampling rate: {SAMPLING_RATE} Hz")
print(f"   Available methods: Threshold-based, PyWaveClus-inspired")


In [ ]:
# Import spike detection module
from utils.spike_detection import SpikeDetector, create_raster_comparison

# Initialize spike detector
spike_detector = SpikeDetector(sampling_rate=SAMPLING_RATE)

print("🔬 Spike Detection Comparison Module Loaded!")
print(f"   Sampling rate: {SAMPLING_RATE} Hz")
print(f"   Available methods: Threshold-based, PyWaveClus-inspired")


In [ ]:
# 🎯 Behavior-Raster-PSTH Visualization
# Create a comprehensive 3x1 plot showing behavioral data, spike raster, and PSTH

# Configure visualization parameters
psth_bin_size = 0.01  # 10ms bins
psth_sigma = 0.025    # 25ms smoothing
threshold_multiplier = -2.0  # Spike detection threshold (-4x RMS)

# Create the integrated visualization
plot_behavior_raster_psth(
    trial_data=trial_data,
    spike_channels=SPIKE_CHANNELS,
    trial_number=TRIAL_NUMBER,
    threshold_multiplier=threshold_multiplier,
    psth_bin_size=psth_bin_size,
    psth_sigma=psth_sigma,
    sampling_rate=30000,
    figsize=(15, 12)
)

print(f"📊 Behavior-Raster-PSTH visualization complete!")
print(f"   Parameters: {psth_bin_size*1000:.0f}ms bins, {psth_sigma*1000:.0f}ms smoothing")
print(f"   Spike detection: {threshold_multiplier}x RMS threshold")
print(f"   Channels analyzed: {len(SPIKE_CHANNELS)}")
